# 序列到序列学习（Seq2Seq）

序列到序列（Seq2Seq）是一种用于处理输入和输出均为可变长度序列的模型框架，典型应用是机器翻译。其目标是学习从输入序列 到输出序列的映射关系，其中输入和输出长度可以不同。

Seq2Seq通常基于编码器-解码器架构实现。编码器一般使用循环神经网络（如RNN、GRU或LSTM），按时间步依次读取输入序列，并更新隐状态，
在读取完整个输入序列后，得到最终隐状态，它被视为对整个输入序列的压缩表示（上下文向量）。该向量包含了输入序列的语义信息，并作为解码器的初始状态。

解码器同样是一个循环神经网络，它在每个时间步基于前一个已生成的词元和当前隐状态来预测下一个词元。解码过程从特殊的开始符号 `<bos>` 开始，
模型不断生成词元，直到输出结束符号 `<eos>` 为止，从而得到完整的目标序列。

在训练过程中，通常采用“教师强制”（teacher forcing）策略，即在每个时间步使用真实的目标词元作为输入，而不是模型预测的结果。具体做法是将目标序列右移一位作为解码器输入，例如将 `<bos> y_1 y_2 \ldots y_{T'}` 作为输入，预测 \(y_1 y_2 \ldots y_{T'} <eos>\)。这种方式可以加速收敛并提高训练稳定性。

Seq2Seq模型的核心优势在于结构简单且通用，能够处理不同长度的输入输出序列，但其也存在局限，例如所有输入信息被压缩到一个固定长度的向量中，容易造成信息瓶颈，尤其是在处理长序列时性能下降。这也是后续引入注意力机制（Attention）和Transformer模型的重要动机。

In [70]:
import collections
import math
import torch
from torch import nn

# 编码器（Encoder）

编码器的作用是将长度可变的输入序列压缩为一个固定形状的上下文向量c，从而对整个输入序列的信息进行表示。在Seq2Seq框架中，这一步相当于“理解输入”，即把原始序列转换为一个可以被后续解码器使用的抽象表示。

在具体实现中，通常使用循环神经网络（RNN、GRU或LSTM）来构建编码器。对于输入序列$x_1, \ldots, x_T$，首先将每个词元转换为对应的特征向量，然后在每个时间步通过递归方式更新隐状态：

$$\mathbf{h}_t = f(\mathbf{x}_t, \mathbf{h}_{t-1}). $$

其中$\mathbf{h}_t$表示当前时间步的隐状态，它逐步累积输入序列的信息。经过最后一个时间步后，可以得到一系列隐状态。

编码器的最终输出是上下文向量，它由所有隐状态通过某个函数得到：
$$\mathbf{c} =  q(\mathbf{h}_1, \ldots, \mathbf{h}_T).$$
在最常见的实现中，直接取最后一个隐状态作为上下文表示，这意味着编码器将整个输入序列的信息压缩到了最后一个时间步的隐状态中。

需要注意的是，如果使用的是单向循环神经网络，那么每个隐状态只依赖于当前位置之前的输入。而如果使用双向循环神经网络，则每个隐状态同时依赖于前后两个方向的序列信息，从而能够编码整个输入序列的全局上下文信息，使表示更加充分。

在实现层面，编码器通常会在最前面加入一个嵌入层（Embedding Layer），将离散的词元索引映射为连续的向量表示。随后，这些向量作为输入送入多层循环神经网络（如多层GRU），逐步生成隐状态并最终得到上下文向量。

In [71]:
import torch
from torch import nn

class Seq2SeqEncoder(nn.Module):
    """用于序列到序列学习的循环神经网络编码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0):
        super(Seq2SeqEncoder, self).__init__()

        # 嵌入层：把token index → 向量
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # GRU编码器
        self.rnn = nn.GRU(
            input_size=embed_size,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            dropout=dropout
        )

    def forward(self, X):
        """
        X: (batch_size, num_steps)
        """

        # → (batch_size, num_steps, embed_size)
        X = self.embedding(X)

        # → (num_steps, batch_size, embed_size)
        X = X.permute(1, 0, 2)

        # RNN计算
        output, state = self.rnn(X)

        # output: (num_steps, batch_size, num_hiddens)
        # state:  (num_layers, batch_size, num_hiddens)

        return output, state

下面实例化上述编码器的实现

In [72]:
encoder = Seq2SeqEncoder(vocab_size=10, embed_size=8, num_hiddens=16,
                         num_layers=2)
encoder.eval()
X = torch.zeros((4, 7), dtype=torch.long)
output, state = encoder(X)
output.shape

torch.Size([7, 4, 16])

# 解码器（Decoder）

解码器的任务是在给定编码器输出的上下文向量c的基础上，逐步生成目标序列。对于输出序列，解码器在每个时间步 建模如下条件概率：
$P(y_{t'} \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$。

这意味着当前输出不仅依赖于之前已经生成的词元，还依赖于编码器提供的全局语义信息。

在实现上，解码器通常也是一个循环神经网络（如RNN/GRU/LSTM）。在时间步t，解码器接收三个信息：上一时刻的输出 、上下文向量、以及上一隐状态，并更新当前隐状态：
$$\mathbf{s}_{t^\prime} = g(y_{t^\prime-1}, \mathbf{c}, \mathbf{s}_{t^\prime-1}).$$
:eqlabel:`eq_seq2seq_s_t`
其中函数g表示解码器的循环更新过程。得到隐状态后，通过输出层（通常是全连接层 + softmax）计算当前词元的概率分布：
$P(y_{t^\prime} \mid y_1, \ldots, y_{t^\prime-1}, \mathbf{c})$。

在初始化阶段，解码器的初始隐状态通常直接使用编码器最后一个时间步的隐状态，这要求编码器和解码器在结构上具有相同的层数和隐藏单元数。为了更充分利用输入序列的信息，常见做法是在每个时间步将上下文向量与当前输入（如词向量）进行拼接，再送入循环神经网络中。

总体而言，解码器通过“递归生成”的方式，从 `<bos>` 开始，逐步生成词元，直到输出 `<eos>` 结束，实现从上下文表示到目标序列的映射。

In [73]:
import torch
from torch import nn

class Seq2SeqDecoder(nn.Module):
    """用于序列到序列学习的循环神经网络解码器"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0):
        super(Seq2SeqDecoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            input_size=embed_size + num_hiddens,
            hidden_size=num_hiddens,
            num_layers=num_layers,
            dropout=dropout
        )
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs):
        # enc_outputs 一般是 (output, state)
        # 这里直接取编码器最后的隐藏状态作为解码器初始状态
        return enc_outputs[1]

    def forward(self, X, state):
        # X: (batch_size, num_steps)
        # 先做嵌入，再调整为 (num_steps, batch_size, embed_size)
        X = self.embedding(X).permute(1, 0, 2)

        # state[-1]: (batch_size, num_hiddens)
        # 扩展成 (num_steps, batch_size, num_hiddens)
        context = state[-1].repeat(X.shape[0], 1, 1)

        # 在特征维拼接输入和上下文
        X_and_context = torch.cat((X, context), dim=2)

        # output: (num_steps, batch_size, num_hiddens)
        # state:  (num_layers, batch_size, num_hiddens)
        output, state = self.rnn(X_and_context, state)

        # 经过全连接层映射到词表维度
        # 再转回 (batch_size, num_steps, vocab_size)
        output = self.dense(output).permute(1, 0, 2)

        return output, state

实例化解码器

In [74]:
decoder = Seq2SeqDecoder(vocab_size=10, embed_size=8, num_hiddens=16,
                         num_layers=2)
decoder.eval()
state = decoder.init_state(encoder(X))
output, state = decoder(X, state)
output.shape, state.shape

(torch.Size([4, 7, 10]), torch.Size([2, 4, 16]))

损失函数

在Seq2Seq模型中，解码器在每个时间步都会输出一个词元的概率分布，通常通过softmax得到，并使用交叉熵损失函数进行优化。但由于实际训练时会对不同长度的序列进行填充（padding）以形成统一的小批量，这些填充位置并不包含真实信息，因此不应参与损失计算。为了解决这个问题，可以使用掩码（mask）机制，例如`sequence_mask`函数，将超出有效长度的部分置为0，从而在计算损失时自动忽略这些无效位置，避免它们对模型训练产生干扰，使模型只关注真实有效的序列部分。

In [75]:
def sequence_mask(X, valid_len, value=0):
    """在序列中屏蔽不相关的项"""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X

X = torch.tensor([[1, 2, 3], [4, 5, 6]])
sequence_mask(X, torch.tensor([1, 2]))

tensor([[1, 0, 0],
        [4, 5, 0]])

In [76]:
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)

        # mask
        for i, l in enumerate(valid_len):
            weights[i, int(l):] = 0

        self.reduction = 'none'
        unweighted_loss = super().forward(
            pred.permute(0, 2, 1), label
        )

        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss

还可以使用此函数屏蔽最后几个轴上的所有项

In [77]:
X = torch.ones(2, 3, 4)
sequence_mask(X, torch.tensor([1, 2]), value=-1)

tensor([[[ 1.,  1.,  1.,  1.],
         [-1., -1., -1., -1.],
         [-1., -1., -1., -1.]],

        [[ 1.,  1.,  1.,  1.],
         [ 1.,  1.,  1.,  1.],
         [-1., -1., -1., -1.]]])

通过扩展softmax交叉熵损失函数来遮蔽不相关的预测。

最初，所有预测词元的掩码都设置为1。 一旦给定了有效长度，与填充词元对应的掩码将被设置为0。 最后，将所有词元的损失乘以掩码，以过滤掉损失中填充词元产生的不相关预测。

In [78]:
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带遮蔽的softmax交叉熵损失函数"""
    # pred的形状：(batch_size,num_steps,vocab_size)
    # label的形状：(batch_size,num_steps)
    # valid_len的形状：(batch_size,)
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)
        self.reduction='none'
        unweighted_loss = super(MaskedSoftmaxCELoss, self).forward(
            pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss

创建三个相同的序列来进行代码健全性检查， 然后分别指定这些序列的有效长度为 4 、 2 和 0 。

In [79]:
loss = MaskedSoftmaxCELoss()
loss(torch.ones(3, 4, 10), torch.ones((3, 4), dtype=torch.long),
     torch.tensor([4, 2, 0]))

tensor([2.3026, 1.1513, 0.0000])

训练

在下面的循环训练过程中，如 :numref:`fig_seq2seq`所示，
特定的序列开始词元（“&lt;bos&gt;”）和
原始的输出序列（不包括序列结束词元“&lt;eos&gt;”）
拼接在一起作为解码器的输入。

In [80]:
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()

    net.to(device)
    net.train()

    for epoch in range(num_epochs):
        total_loss = 0.0
        total_tokens = 0

        for X, X_valid_len, Y, Y_valid_len in data_iter:
            X = X.to(device)
            Y = Y.to(device)
            Y_valid_len = Y_valid_len.to(device)

            bos = torch.tensor(
                [tgt_vocab['<bos>']] * Y.shape[0],
                device=device
            ).reshape(-1, 1)

            dec_input = torch.cat([bos, Y[:, :-1]], dim=1)

            Y_hat, _ = net(X, dec_input)

            l = loss(Y_hat, Y, Y_valid_len)   # shape: (batch_size,)

            optimizer.zero_grad()
            l.sum().backward()
            optimizer.step()

            total_loss += l.sum().item()
            total_tokens += Y_valid_len.sum().item()

        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1}, loss {total_loss / total_tokens:.4f}")

In [81]:
import requests
import zipfile
import io

def read_data_nmt():
    url = "http://d2l-data.s3-accelerate.amazonaws.com/fra-eng.zip"

    # 下载
    response = requests.get(url)

    # 解压
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        with z.open("fra-eng/fra.txt") as f:
            return f.read().decode("utf-8")

In [83]:
import re
import torch
from torch.utils.data import DataLoader, TensorDataset

def preprocess_nmt(text):
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    out = []
    for i, char in enumerate(text):
        if char in ',.!?':
            if i > 0 and text[i - 1] != ' ':
                out.append(' ')
        out.append(char)
    return ''.join(out)

def tokenize_nmt(text, num_examples=None):
    source, target = [], []
    for i, line in enumerate(text.split('\n')):
        if num_examples is not None and i >= num_examples:
            break
        parts = line.split('\t')
        if len(parts) == 2:
            source.append(parts[0].split())
            target.append(parts[1].split())
    return source, target

class Vocab:
    def __init__(self, tokens, min_freq=2, reserved_tokens=None):
        if reserved_tokens is None:
            reserved_tokens = []

        counter = {}
        for line in tokens:
            for token in line:
                counter[token] = counter.get(token, 0) + 1

        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: i for i, token in enumerate(self.idx_to_token)}

        for token, freq in counter.items():
            if freq >= min_freq and token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if isinstance(tokens, (list, tuple)):
            return [self.__getitem__(token) for token in tokens]
        return self.token_to_idx.get(tokens, self.token_to_idx['<unk>'])
    def to_tokens(self, indices):
      if isinstance(indices, (list, tuple)):
        return [self.idx_to_token[int(i)] for i in indices]
      return self.idx_to_token[int(indices)]

def truncate_pad(line, num_steps, padding_token):
    if len(line) > num_steps:
        return line[:num_steps]
    return line + [padding_token] * (num_steps - len(line))

def build_array(lines, vocab, num_steps):
    lines = [vocab[line] + [vocab['<eos>']] for line in lines]
    array = torch.tensor(
        [truncate_pad(line, num_steps, vocab['<pad>']) for line in lines],
        dtype=torch.long
    )
    valid_len = (array != vocab['<pad>']).sum(dim=1)
    return array, valid_len

def load_data_nmt(batch_size, num_steps, num_examples=600):
    text = preprocess_nmt(read_data_nmt())
    source, target = tokenize_nmt(text, num_examples)

    print('source样本数:', len(source))
    print('target样本数:', len(target))
    print('source前2个样本:', source[:2])
    print('target前2个样本:', target[:2])

    src_vocab = Vocab(source, min_freq=2,
                      reserved_tokens=['<pad>', '<bos>', '<eos>'])
    tgt_vocab = Vocab(target, min_freq=2,
                      reserved_tokens=['<pad>', '<bos>', '<eos>'])

    src_array, src_valid_len = build_array(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array(target, tgt_vocab, num_steps)

    print('src_array.shape =', src_array.shape)
    print('tgt_array.shape =', tgt_array.shape)

    dataset = TensorDataset(src_array, src_valid_len, tgt_array, tgt_valid_len)
    train_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    return train_iter, src_vocab, tgt_vocab

In [84]:
import torch
from torch import nn

# 超参数
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs = 0.005, 300
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 你前面已经自己实现过的机器翻译数据加载函数
train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)

# 编码器
encoder = Seq2SeqEncoder(
    len(src_vocab), embed_size, num_hiddens, num_layers, dropout
)

# 解码器
decoder = Seq2SeqDecoder(
    len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout
)

# Encoder-Decoder 总模型
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_outputs, *args)
        return self.decoder(dec_X, dec_state)

net = EncoderDecoder(encoder, decoder).to(device)

# 训练
train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

source样本数: 600
target样本数: 600
source前2个样本: [['go', '.'], ['hi', '.']]
target前2个样本: [['va', '!'], ['salut', '!']]
src_array.shape = torch.Size([600, 10])
tgt_array.shape = torch.Size([600, 10])
epoch 10, loss 0.2196
epoch 20, loss 0.1648
epoch 30, loss 0.1298
epoch 40, loss 0.1042
epoch 50, loss 0.0834
epoch 60, loss 0.0709
epoch 70, loss 0.0605
epoch 80, loss 0.0528
epoch 90, loss 0.0472
epoch 100, loss 0.0415
epoch 110, loss 0.0379
epoch 120, loss 0.0346
epoch 130, loss 0.0335
epoch 140, loss 0.0307
epoch 150, loss 0.0286
epoch 160, loss 0.0273
epoch 170, loss 0.0260
epoch 180, loss 0.0260
epoch 190, loss 0.0246
epoch 200, loss 0.0245
epoch 210, loss 0.0241
epoch 220, loss 0.0227
epoch 230, loss 0.0223
epoch 240, loss 0.0223
epoch 250, loss 0.0211
epoch 260, loss 0.0209
epoch 270, loss 0.0206
epoch 280, loss 0.0208
epoch 290, loss 0.0211
epoch 300, loss 0.0206


预测

采用一个接着一个词元的方式预测输出序列，
每个解码器当前时间步的输入都将来自于前一时间步的预测词元。

使用循环神经网络编码器-解码器逐词元地预测输出序列。

In [85]:
import torch

def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab, num_steps,
                    device, save_attention_weights=False):
    """序列到序列模型的预测"""
    net.eval()

    # 源句子分词并转索引
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [src_vocab['<eos>']]
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)

    # 截断或填充
    src_tokens = truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])

    # 添加 batch 维度: (1, num_steps)
    enc_X = torch.tensor(src_tokens, dtype=torch.long, device=device).unsqueeze(0)

    # 编码
    try:
        enc_outputs = net.encoder(enc_X, enc_valid_len)
    except TypeError:
        enc_outputs = net.encoder(enc_X)

    # 初始化解码器状态
    try:
        dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    except TypeError:
        dec_state = net.decoder.init_state(enc_outputs)

    # 解码器初始输入 <bos>
    dec_X = torch.tensor([[tgt_vocab['<bos>']]], dtype=torch.long, device=device)

    output_seq = []
    attention_weight_seq = []

    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)

        # 取当前时间步预测概率最大的词元
        dec_X = Y.argmax(dim=2)
        pred = int(dec_X.squeeze(0).item())

        if save_attention_weights and hasattr(net.decoder, "attention_weights"):
            attention_weight_seq.append(net.decoder.attention_weights)

        # 遇到 <eos> 就停止
        if pred == tgt_vocab['<eos>']:
            break

        output_seq.append(pred)

    return ' '.join(tgt_vocab.to_tokens(output_seq)), attention_weight_seq

## 预测序列的评估

我们可以通过与真实的标签序列进行比较来评估预测序列。

BLEU（bilingual evaluation understudy）
最先是用于评估机器翻译的结果，
但现在它已经被广泛用于测量许多应用的输出序列的质量。
BLEU定义为：

$$ \exp\left(\min\left(0, 1 - \frac{\mathrm{len}_{\text{label}}}{\mathrm{len}_{\text{pred}}}\right)\right) \prod_{n=1}^k p_n^{1/2^n},$$
:eqlabel:`eq_bleu`

其中$\mathrm{len}_{\text{label}}$表示标签序列中的词元数和
$\mathrm{len}_{\text{pred}}$表示预测序列中的词元数，
$k$是用于匹配的最长的$n$元语法。

当预测序列与标签序列完全相同时，BLEU为$1$。
此外，由于$n$元语法越长则匹配难度越大，
所以BLEU为更长的$n$元语法的精确度分配更大的权重。

In [86]:
import math
import collections

def bleu(pred_seq, label_seq, k):
    """计算BLEU分数"""
    pred_tokens = pred_seq.split(' ')
    label_tokens = label_seq.split(' ')

    len_pred, len_label = len(pred_tokens), len(label_tokens)

    # 防止空序列报错
    if len_pred == 0:
        return 0.0

    # brevity penalty（长度惩罚）
    score = math.exp(min(0, 1 - len_label / len_pred))

    for n in range(1, k + 1):
        num_matches = 0
        label_subs = collections.defaultdict(int)

        # 统计label的n-gram
        for i in range(len_label - n + 1):
            key = ' '.join(label_tokens[i: i + n])
            label_subs[key] += 1

        # 匹配pred的n-gram
        for i in range(len_pred - n + 1):
            key = ' '.join(pred_tokens[i: i + n])
            if label_subs[key] > 0:
                num_matches += 1
                label_subs[key] -= 1

        denom = (len_pred - n + 1)

        # 防止除0（非常关键）
        if denom == 0 or num_matches == 0:
            return 0.0

        score *= math.pow(num_matches / denom, math.pow(0.5, n))

    return score

利用训练好的循环神经网络“编码器－解码器”模型，将几个英语句子翻译成法语，并计算BLEU的最终结果。

In [87]:
engs = ['go .', "i lost .", 'he\'s calm .', 'i\'m home .']
fras = ['va !', 'j\'ai perdu .', 'il est calme .', 'je suis chez moi .']
for eng, fra in zip(engs, fras):
    translation, attention_weight_seq = predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device)
    print(f'{eng} => {translation}, bleu {bleu(translation, fra, k=2):.3f}')

go . => va !, bleu 1.000
i lost . => j'ai <unk> quoi faire ?, bleu 0.000
he's calm . => il <unk> est pas venu venu suis-je venu suis-je suis-je, bleu 0.000
i'm home . => je suis chez la mouillé ., bleu 0.649
